# Data Agent Platform Studio — Colab GPU / CPU

A self-contained Data Agent Studio using one shared Bronze/Silver/Gold pipeline and two CrewAI agents. Both agents share one local Hugging Face model. CUDA (T4 target) is recommended; CPU-only inference is also supported (slower, FP32).

No external hosted-agent or vendor-agent API is used. Select a GPU runtime when available, or use the standard CPU runtime. Set `HF_DEVICE=cpu` to force CPU or `HF_DEVICE=auto` to prefer CUDA and fall back to CPU.

In [ ]:
%cd /content
!if [ ! -d data-agent-platform-studio/.git ]; then git clone -b feature/colab-crewai-dual-agent-studio https://github.com/msellamiTN/data-agent-platform-studio.git; fi
%cd /content/data-agent-platform-studio

In [ ]:
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet --no-cache-dir --force-reinstall 'Pillow==11.3.0'
!python -m pip install --quiet -r requirements-colab.txt
!python -m pip check
from PIL import Image, ImageColor
from crewai import Agent, Crew, Process, Task
from crewai.tools import tool
import gradio, transformers, pandas, torch
import PIL
print('Dependency smoke check passed')
print('Pillow:', PIL.__version__)
print('Gradio:', gradio.__version__)
print('Transformers:', transformers.__version__)
print('PyTorch:', torch.__version__)


In [ ]:
import os, platform, torch
from packaging.version import Version
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if Version(torch.__version__.split('+')[0]) < Version('2.5.0'):
    raise RuntimeError('PyTorch >= 2.5 is required. Select a current Colab runtime; do not blindly replace its CUDA-specific wheel.')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    os.environ.setdefault('HF_DEVICE', 'auto')
else:
    os.environ['HF_DEVICE'] = 'cpu'
    print('Using CPU inference (FP32). Expect slower generation and higher RAM use.')
    print('CPU threads:', os.getenv('HF_CPU_THREADS', '4'))


In [ ]:
import os
from studio.config import MODEL_ID
print('Hugging Face model:', MODEL_ID)
print('Selected device:', os.getenv('HF_DEVICE', 'auto'))
print('Precision: FP16 on CUDA; FP32 on CPU')
print('The model is downloaded lazily on the first agent execution.')

## Launch the web Studio

Run the Data Engineering Agent first (upload a CSV or use the Retail Sales demo). Then switch to Data Analysis and ask a question about the same active Gold dataset. The Gradio share URL is accessible to anyone who obtains it: use synthetic data in this lab.

In [ ]:
from app import demo
demo.queue(default_concurrency_limit=1).launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    show_error=True,
)